In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# imports + file path

from pathlib import Path
import pandas as pd

# file path in your Google Drive
INPUT_FILE = "/content/drive/MyDrive/Math 2800/Data/mxmh_survey_results.csv"

OUTPUT_FILE = "/content/drive/MyDrive/Math 2800/Data/mxmh_survey_results_cleaned.csv"
REPORT_FILE = "/content/drive/MyDrive/Math 2800/Data/data_cleaning_report.txt"

In [ ]:
# loading data + inital exploration

df_raw = pd.read_csv(INPUT_FILE)
df = df_raw.copy()

print("Initial shape:", df.shape)

print("\nInitial info:")
df.info()

print("\nInitial summary stats:")
display(df.describe(include="all"))

print("\nInitial missing values:")
display(df.isnull().sum())

In [ ]:
# helper functions for cleaning

def normalize_text(series):
    return series.astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

def compare_missingness(original_df, target_col, compare_cols):
    missing_group = original_df[original_df[target_col].isna()]
    non_missing_group = original_df[original_df[target_col].notna()]

    print(f"\n--- Missingness check for: {target_col} ---")
    print(f"Missing rows: {len(missing_group)}")
    print(f"Non-missing rows: {len(non_missing_group)}")

    for col in compare_cols:
        if col in original_df.columns and pd.api.types.is_numeric_dtype(original_df[col]):
            miss_mean = missing_group[col].mean()
            nonmiss_mean = non_missing_group[col].mean()
            print(f"{col}: missing mean = {miss_mean:.3f}, non-missing mean = {nonmiss_mean:.3f}")

In [ ]:
# fairness checks before cleaning

compare_missingness(df_raw, "BPM", ["Age", "Hours per day", "Anxiety", "Depression", "Insomnia", "OCD"])
compare_missingness(df_raw, "Music effects", ["Age", "Hours per day", "Anxiety", "Depression", "Insomnia", "OCD"])

In [ ]:
# cleaning

# clean column names
df.columns = df.columns.str.strip()

# numeric conversion
numeric_cols = ["Age", "Hours per day", "BPM", "Anxiety", "Depression", "Insomnia", "OCD"]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# text columns
text_cols = [
    "Primary streaming service",
    "While working",
    "Instrumentalist",
    "Composer",
    "Fav genre",
    "Exploratory",
    "Foreign languages",
    "Music effects",
    "Permissions",
]

for col in text_cols:
    if col in df.columns:
        df[col] = normalize_text(df[col])

# standardization
df["Fav genre"] = df["Fav genre"].str.title()
df["Primary streaming service"] = df["Primary streaming service"].str.title()
df["Music effects"] = df["Music effects"].str.title()

for col in ["While working", "Instrumentalist", "Composer", "Exploratory", "Foreign languages", "Permissions"]:
    if col in df.columns:
        df[col] = df[col].str.title()

In [ ]:
#handle missing data

# drop key missing
key_cols = ["Age", "Hours per day", "Anxiety", "Depression", "Insomnia", "OCD"]
before_drop = len(df)

df = df.dropna(subset=key_cols)

after_drop = len(df)

# fill BPM
df["BPM"] = df["BPM"].fillna(df["BPM"].median())

# fill categorical
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

In [ ]:
# sanity filter for hour slistened
df = df[(df["Hours per day"] >= 0) & (df["Hours per day"] <= 24)]

In [ ]:
# final output

print("After cleaning shape:", df.shape)

print("\nMissing values after cleaning:")
display(df.isnull().sum())

print("\nCleaned summary stats:")
display(df.describe(include="all"))

print("\nRows before:", before_drop)
print("Rows after:", after_drop)
print("Rows removed:", before_drop - after_drop)

In [ ]:
# save files

df.to_csv(OUTPUT_FILE, index=False)

report_lines = [
    "Data Cleaning Report",
    "====================",
    f"Original rows: {len(df_raw)}",
    f"Rows after cleaning: {len(df)}",
    "",
    "steps completed:",
    "- Converted numeric columns",
    "- Standardized text categories",
    "- Dropped rows missing key variables",
    "- Imputed BPM with median",
    "- Filled categorical missing values",
    "",
    "ethical notes:",
    "- Checked missingness before cleaning",
    "- Maintained fairness and representation",
]

with open(REPORT_FILE, "w") as f:
    f.write("\n".join(report_lines))

print("Saved cleaned dataset and report.")